<a href="https://colab.research.google.com/github/Sahazid/Youtbue_Scrapping_Code/blob/main/YoutubeScrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gspread gspread_dataframe pandas requests


In [ ]:
from google.colab import auth
auth.authenticate_user() #It will take the Access Permission make it allow;

In [ ]:
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)


In [ ]:
spreadsheet = gc.open_by_key(" ") #Paste the SpreadSheet id ;
worksheet = spreadsheet.sheet1

In [ ]:
import requests

API_KEY = " " #Paste the youtube API Key;
VIDEO_ID = " " # Paste the youtube video id for scrapping comments;

In [ ]:
video_url = "https://www.googleapis.com/youtube/v3/videos"

params = {
    "part": "snippet",
    "id": VIDEO_ID,
    "key": API_KEY
}

video_data = requests.get(video_url, params=params).json()
video_title = video_data["items"][0]["snippet"]["title"]

In [ ]:
url = "https://www.googleapis.com/youtube/v3/commentThreads"

params = {
    "part": "snippet,replies",
    "videoId": VIDEO_ID,
    "maxResults": 100,
    "key": API_KEY
}

rows = []
comment_counter = 1
comment_count = 0
MAX_COMMENTS = 300

while True:

    response = requests.get(url, params=params).json()

    for item in response["items"]:

        if comment_count >= MAX_COMMENTS:
            break

        top = item["snippet"]["topLevelComment"]["snippet"]

        root_id = f"C{comment_counter:03}"

        rows.append({
            "Post title": video_title,
            "Comment_ID": root_id,
            "Parent_ID": "NULL",
            "Level": 1,
            "Root_ID": root_id,
            "Text": top["textDisplay"]
        })

        comment_counter += 1
        comment_count += 1

        if "replies" in item:

            for reply in item["replies"]["comments"]:

                if comment_count >= MAX_COMMENTS:
                    break

                r = reply["snippet"]
                reply_id = f"C{comment_counter:03}"

                rows.append({
                    "Post title": "",
                    "Comment_ID": reply_id,
                    "Parent_ID": root_id,
                    "Level": 2,
                    "Root_ID": root_id,
                    "Text": r["textDisplay"]
                })

                comment_counter += 1
                comment_count += 1

    if comment_count >= MAX_COMMENTS:
        break

    if "nextPageToken" not in response:
        break

    params["pageToken"] = response["nextPageToken"]

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)

In [ ]:
from gspread_dataframe import set_with_dataframe

existing = worksheet.get_all_values()

if len(existing) == 0:
    worksheet.append_row(df.columns.tolist())

worksheet.append_rows(df.values.tolist())